# Buổi 8: Mô hình phân lớp và phân cụm
**Môn học:** Python cho Phân tích Dữ liệu Kinh doanh · **BÀI TẬP THỰC HÀNH**

**Mục tiêu:**
- Xây dựng mô hình phân lớp (Logistic Regression, Decision Tree, Random Forest) dự đoán khách hàng rời bỏ
- Đánh giá bằng ma trận nhầm lẫn, accuracy, precision, recall, F1 và chọn ngưỡng theo mục tiêu kinh doanh
- Phân khúc khách hàng bằng K-Means; chọn số cụm bằng Elbow và Silhouette
- Mô tả, đặt tên phân khúc và đề xuất hành động

**Dữ liệu dùng trong buổi:** `telco_churn.csv`, `ecommerce_orders_clean.csv` (trong thư mục `data/`). Nếu dùng Google Colab, tải thư mục `data/` lên rồi sửa biến `DATA` bên dưới.

**Cấu trúc:** Phần A – bài tập có hướng dẫn (làm theo, chạy từng ô) · Phần B – bài tập tự làm (bắt buộc, nộp cuối buổi) · Phần C – bài tập làm thêm (dành cho bạn muốn thử thách).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report)
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
DATA = "../data/"
sns.set_theme(style="whitegrid")

## Phần A. Bài tập có hướng dẫn

### A1. Bài toán phân lớp: dự đoán khách hàng rời bỏ (churn)
Nhà mạng có dữ liệu 2.000 thuê bao. Biến `roi_bo` = 1 nếu khách đã hủy dịch vụ. Mục tiêu: tìm khách có nguy cơ rời bỏ
để chăm sóc trước.

In [ ]:
ch = pd.read_csv(DATA + "telco_churn.csv")
print(ch.shape); print("Tỷ lệ rời bỏ:", ch["roi_bo"].mean().round(3))
ch.head()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
for a, c in zip(ax, ["loai_hop_dong", "dich_vu_internet", "phuong_thuc_tt"]):
    (ch.groupby(c)["roi_bo"].mean() * 100).sort_values().plot(kind="barh", ax=a, color="#1f6feb")
    a.set(title=f"Tỷ lệ rời bỏ (%) theo {c}", ylabel="")
plt.tight_layout(); plt.show()

### A2. Chuẩn bị dữ liệu

In [ ]:
X = pd.get_dummies(ch.drop(columns=["ma_kh", "roi_bo"]), drop_first=True, dtype=int)
y = ch["roi_bo"]
# stratify=y giữ tỷ lệ rời bỏ như nhau ở tập train và test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
scaler = StandardScaler().fit(X_train)            # chuẩn hóa cho Logistic Regression
X_train_s, X_test_s = scaler.transform(X_train), scaler.transform(X_test)
print(X.columns.tolist())

### A3. Huấn luyện ba mô hình

In [ ]:
mo_hinh = {
    "Logistic Regression": (LogisticRegression(max_iter=1000), True),
    "Decision Tree":       (DecisionTreeClassifier(max_depth=4, random_state=42), False),
    "Random Forest":       (RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42), False),
}
ket_qua = []
for ten, (m, can_scale) in mo_hinh.items():
    Xtr, Xte = (X_train_s, X_test_s) if can_scale else (X_train, X_test)
    m.fit(Xtr, y_train)
    p = m.predict(Xte)
    ket_qua.append({"Mô hình": ten, "Accuracy": accuracy_score(y_test, p), "Precision": precision_score(y_test, p),
                    "Recall": recall_score(y_test, p), "F1": f1_score(y_test, p)})
pd.DataFrame(ket_qua).round(3)

### A4. Hiểu các chỉ số qua ma trận nhầm lẫn
- **Precision** – trong số khách mô hình *báo sẽ rời bỏ*, bao nhiêu % thực sự rời bỏ (chi phí gọi nhầm)
- **Recall** – trong số khách *thực sự rời bỏ*, mô hình bắt được bao nhiêu % (cơ hội bỏ lỡ)
- Dữ liệu mất cân bằng (≈30% rời bỏ) → **accuracy** dễ gây hiểu lầm; nên xem precision/recall/F1.

In [ ]:
lg = mo_hinh["Logistic Regression"][0]
ConfusionMatrixDisplay.from_estimator(lg, X_test_s, y_test, display_labels=["Ở lại", "Rời bỏ"], cmap="Blues")
plt.title("Ma trận nhầm lẫn – Logistic Regression"); plt.show()
print(classification_report(y_test, lg.predict(X_test_s), target_names=["Ở lại", "Rời bỏ"]))

In [ ]:
# Cây quyết định dễ giải thích cho nhà quản lý
dt = mo_hinh["Decision Tree"][0]
plt.figure(figsize=(18, 7))
plot_tree(dt, feature_names=X.columns, class_names=["Ở lại", "Rời bỏ"], filled=True, max_depth=2, fontsize=9)
plt.show()

In [ ]:
# Yếu tố ảnh hưởng: hệ số Logistic (dữ liệu đã chuẩn hóa nên so sánh được độ lớn)
pd.Series(lg.coef_[0], index=X.columns).sort_values().plot(kind="barh", title="Hệ số Logistic Regression (+ tăng nguy cơ rời bỏ)")
plt.show()

### A5. Chọn ngưỡng xác suất theo mục tiêu kinh doanh
`predict_proba` trả về xác suất rời bỏ. Mặc định ngưỡng 0,5; hạ ngưỡng → bắt được nhiều khách hơn (recall ↑) nhưng gọi nhầm nhiều hơn (precision ↓).

In [ ]:
xs = lg.predict_proba(X_test_s)[:, 1]
bang = []
for nguong in [0.2, 0.3, 0.4, 0.5, 0.6]:
    p = (xs >= nguong).astype(int)
    bang.append({"Ngưỡng": nguong, "Số KH cần gọi": p.sum(), "Precision": precision_score(y_test, p),
                 "Recall": recall_score(y_test, p)})
pd.DataFrame(bang).round(3)

### A6. Bài toán phân cụm: phân khúc khách hàng bằng K-Means
Phân cụm là học **không giám sát**: không có nhãn đúng/sai, mô hình tự gom khách giống nhau. Dùng chỉ số RFM (buổi 5).
K-Means dựa trên khoảng cách → cần **chuẩn hóa** để các biến có thang đo như nhau.

In [ ]:
od = pd.read_csv(DATA + "ecommerce_orders_clean.csv", parse_dates=["ngay_dat"])
od = od[od["trang_thai"] == "Hoàn thành"].copy()
od["doanh_thu"] = od["so_luong"] * od["don_gia"]
moc = od["ngay_dat"].max() + pd.Timedelta(days=1)
rfm = od.groupby("ma_kh").agg(R=("ngay_dat", lambda x: (moc - x.max()).days),
                              F=("ma_don", "nunique"), M=("doanh_thu", "sum"))
rfm["M_log"] = np.log1p(rfm["M"])        # log giảm ảnh hưởng của khách chi tiêu rất lớn
Z = StandardScaler().fit_transform(rfm[["R", "F", "M_log"]])
rfm.describe().round(1)

In [ ]:
# Chọn số cụm k: Elbow (inertia) và Silhouette (càng cao càng tốt)
ks = range(2, 9)
inertia, sil = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Z)
    inertia.append(km.inertia_); sil.append(silhouette_score(Z, km.labels_))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(ks, inertia, marker="o"); ax[0].set(title="Elbow", xlabel="k", ylabel="Inertia")
ax[1].plot(ks, sil, marker="o", color="#2da44e"); ax[1].set(title="Silhouette", xlabel="k")
plt.show()

In [ ]:
km = KMeans(n_clusters=4, n_init=10, random_state=42).fit(Z)
rfm["cum"] = km.labels_
mo_ta = rfm.groupby("cum").agg(so_kh=("R", "size"), R_tb=("R", "mean"), F_tb=("F", "mean"),
                               M_tb_trieu=("M", lambda x: x.mean() / 1e6))
mo_ta["ty_trong_dt_%"] = rfm.groupby("cum")["M"].sum() / rfm["M"].sum() * 100
mo_ta.round(1).sort_values("M_tb_trieu", ascending=False)

In [ ]:
sns.scatterplot(data=rfm, x="R", y="F", hue="cum", size="M", sizes=(10, 200), palette="tab10", alpha=0.7)
plt.title("Phân cụm khách hàng theo Recency và Frequency (kích thước = chi tiêu)"); plt.show()

**Đặt tên phân khúc** dựa trên bảng mô tả (ví dụ): R thấp – F cao – M cao → *VIP*; R thấp – F trung bình → *Trung thành*;
R thấp – F thấp → *Khách mới*; R cao → *Ngủ đông*. Số thứ tự cụm có thể khác nhau giữa các lần chạy – luôn đặt tên dựa trên đặc điểm.

## Phần B. Bài tập tự làm

### Bài B1. Mô hình rời bỏ với tập biến rút gọn

Chỉ dùng 3 biến `thoi_gian_su_dung_thang`, `cuoc_thang_nghin`, `so_lan_goi_ho_tro` cùng loại hợp đồng (one-hot).
Huấn luyện Random Forest và so sánh F1, Recall với mô hình đầy đủ ở Phần A. Bớt biến có làm giảm chất lượng nhiều không?

In [ ]:
# Viết code của bạn ở đây


### Bài B2. Mức độ quan trọng của biến

Từ mô hình Random Forest đầy đủ ở Phần A, vẽ biểu đồ `feature_importances_` (10 biến quan trọng nhất).
Viết 2–3 câu: ba yếu tố nào ảnh hưởng mạnh nhất đến việc rời bỏ và gợi ý hành động cho mỗi yếu tố.

In [ ]:
# Viết code của bạn ở đây


### Bài B3. Danh sách khách cần chăm sóc

Dùng Logistic Regression dự đoán xác suất rời bỏ cho **toàn bộ** 2.000 khách (nhớ chuẩn hóa bằng `scaler`).
Lấy 50 khách đang ở lại (`roi_bo == 0`) có xác suất rời bỏ cao nhất, xuất ra `khach_can_cham_soc.csv`
gồm mã KH, loại hợp đồng, cước tháng, xác suất.

In [ ]:
# Viết code của bạn ở đây


### Bài B4. Phân cụm với số cụm khác

Chạy lại K-Means với k = 3 và k = 5 trên ma trận `Z`. So sánh silhouette và bảng mô tả cụm.
Theo bạn, k nào hữu ích nhất cho bộ phận marketing? Vì sao?

In [ ]:
# Viết code của bạn ở đây


### Bài B5. Đặt tên phân khúc và đề xuất chính sách

Với kết quả k = 4 ở Phần A, tạo cột `phan_khuc` gán tên cho từng cụm dựa trên bảng mô tả (dùng `map` với dict
`{so_cum: "tên"}`). Lập bảng gồm: phân khúc, số khách, chi tiêu TB, tỷ trọng doanh thu, và cột `hanh_dong`
đề xuất một chính sách cho mỗi phân khúc.

In [ ]:
# Viết code của bạn ở đây


## Phần C. Bài tập làm thêm

### Bài C1. Bài toán lợi nhuận khi chọn ngưỡng

Giả sử: gọi chăm sóc một khách tốn 50 nghìn đồng; nếu khách đó thực sự định rời bỏ, chăm sóc giữ chân được 40% và mỗi khách
giữ được mang lại 1,2 triệu đồng. Với các ngưỡng 0,1 → 0,9 (bước 0,05), tính lợi nhuận kỳ vọng trên tập test và vẽ đồ thị.
Ngưỡng nào tối ưu?

In [ ]:
# Viết code của bạn ở đây


### Bài C2. Tinh chỉnh tham số bằng GridSearchCV

Dùng `GridSearchCV` tìm `max_depth` ∈ {3, 5, 8, None} và `n_estimators` ∈ {100, 300} cho Random Forest, tiêu chí `f1`, cv=5.
Báo cáo tham số tốt nhất và F1 trên tập test.

In [ ]:
# Viết code của bạn ở đây


### Bài C3. Mất cân bằng lớp

Huấn luyện lại Logistic Regression với `class_weight="balanced"`. So sánh precision, recall với mô hình gốc.
Khi nào nên dùng tùy chọn này trong kinh doanh?

In [ ]:
# Viết code của bạn ở đây
